# Verify the ProprioTunedCNN sensorprocessing

Reload a pre-trained model specified by an experiment trained by the Train-ProprioTuned-CNN notebook.

This notebook runs a number of experiments that illustrate the performance. 

In [ ]:
import sys
sys.path.append("..")

from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

import pathlib
import matplotlib.pyplot as plt
import torch
# from torch.utils.data import DataLoader, TensorDataset
from pprint import pprint

from sensorprocessing.sp_factory import create_sp

from demonstration.demonstration import Demonstration
import sensorprocessing.sp_helper as sp_helper


### Exp-run initialization
Create the exp/run-s that describe the parameters of the training. 
Some of the code here is structured in such a way as to make the notebook automatizable with papermill.

In [ ]:
# *** Initialize the variables with default values 
# *** This cell should be tagged as parameters     
# *** If papermill is used, some of the values will be overwritten

# The experiment/run we are going to run: the specified model will be created
experiment = "sensorprocessing_propriotuned_cnn"
# run = "vgg19_128"
# run = "resnet50_128"
# run = "vgg19_256"
run = "resnet50_256"

# If it is set to true, the exprun will be recreated from scratch
creation_style = "exist-ok"
# If not None, set an external experiment path
expruns_path = None
# If not None, set an output path
results_path = None

In [ ]:
# create the necessary exp/run objects

if expruns_path:
    expruns_path = pathlib.Path(expruns_path)
    assert expruns_path.exists()
    Config().set_exprun_path(expruns_path)
if results_path:
    results_path = pathlib.Path(results_path)
    assert results_path.exists()
    Config().set_results_path(results_path)

exp = Config().get_experiment(experiment, run, creation_style=creation_style)
print(exp)

### Create the SensorProcessing object

Check whether we can load back the saved torch model: `create_sp` builds the model selected by the exp/run and loads its weights.

In [ ]:
sp = create_sp(exp)
# the trained model, used below for the encoding
model = sp.enc

In [ ]:
# Run the loaded torch model on all the specified validation data

transform = sp_helper.get_transform_to_sp(exp)

with torch.no_grad():
    for val in exp["validation_data"]:
        run, demo_name, camera = val
        exp_demo = Config().get_experiment("demonstration", run)
        demo = Demonstration(exp_demo, demo_name)
        for i in range(demo.metadata["maxsteps"]):
            sensor_readings, _ = demo.get_image(i, camera=camera, transform=transform)
            z = model.encode(sensor_readings)
            break

print(f"The shape of the output is: {z.shape}")
print(f"One example z looks like this:\n{z}")


### Use the SensorProcessing object

In [ ]:
# run all the validation images through the sensor processing
for val in exp["validation_data"]:
    run, demo_name, camera = val
    exp_demo = Config().get_experiment("demonstration", run)
    demo = Demonstration(exp_demo, demo_name)
    for i in range(demo.metadata["maxsteps"]):
        sensor_readings, _ = demo.get_image(i, camera=camera, transform=transform)
        z = sp.process(sensor_readings)
        break
# print the last z, as an example
print(z)
print(z.shape)

In [ ]:
exp.done()